# 14 · TTF Daily Market Report

Run every morning to produce a one-page summary of TTF market conditions.
Outputs a dated PDF to `data/processed/`.

**Run order:** cells 0 → 6.  
**Requires:** `data/raw/ttf_curve.csv` (notebook 00) and `data/processed/eu_aggregate_full.parquet` (notebook 01).

In [ ]:
import sys, os, warnings, calendar
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import date, timedelta
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f'\u2705 Root: {_c}'); break

# ═══════════════════════════════════════════════════════════════════════════════
REPORT_DATE = None   # None = last available date in ttf_curve.csv
# ═══════════════════════════════════════════════════════════════════════════════

TTF_PATH     = Path('data/raw/ttf_curve.csv')
STORAGE_PATH = Path('data/processed/eu_aggregate_full.parquet')
LNG_PATH     = Path('data/processed/eu_lng_full.parquet')

def _load_ts(path, **kwargs):
    if not path.exists(): return pd.DataFrame()
    if path.suffix == '.csv':
        df = pd.read_csv(path, index_col=0, parse_dates=True, **kwargs)
    else:
        df = pd.read_parquet(path, **kwargs)
    df.index = pd.to_datetime(df.index).tz_localize(None)
    for col in df.columns:
        try: df[col] = pd.to_numeric(df[col], errors='coerce')
        except TypeError: pass
    return df.sort_index()

ttf     = _load_ts(TTF_PATH)
storage = _load_ts(STORAGE_PATH)
lng     = _load_ts(LNG_PATH)

if ttf.empty:     raise FileNotFoundError(f'{TTF_PATH} not found — run notebook 00 first')
if storage.empty: raise FileNotFoundError(f'{STORAGE_PATH} not found — run notebook 01 first')

# Determine report date
if REPORT_DATE is None:
    rdate = ttf.index[-1]
else:
    rdate = pd.Timestamp(REPORT_DATE)
    rdate = ttf.index[ttf.index <= rdate][-1]

ttf_row = ttf.loc[rdate]
m_cols  = sorted([c for c in ttf.columns if c.startswith('M') and c[1:].isdigit()],
                  key=lambda c: int(c[1:]))

def delivery_label(base_ts, m):
    """Delivery month label for tenor M{m} on date base_ts.  M1 = next calendar month."""
    dm = (base_ts.month + m - 1) % 12 + 1
    yr = base_ts.year + (base_ts.month + m - 1) // 12
    return f"{calendar.month_abbr[dm]}'{str(yr)[-2:]}"

print(f'\u2705 TTF      : {ttf.shape}  {ttf.index.min().date()} \u2192 {ttf.index.max().date()}')
print(f'\u2705 Storage  : {storage.shape}  {storage.index.min().date()} \u2192 {storage.index.max().date()}')
print(f'{"\u2705" if not lng.empty else "\u2139\ufe0f"} LNG      : {lng.shape if not lng.empty else "not found (optional)"}')
print(f'\nReport date: {rdate.date()}')

## 1 · Header Snapshot

In [ ]:
# ── TTF prices & changes ──────────────────────────────────────────────────────
prev_row = ttf.loc[ttf.index[ttf.index < rdate][-1]] if (ttf.index < rdate).any() else ttf_row

def price_line(m):
    col = f'M{m}'
    if col not in ttf_row.index or pd.isna(ttf_row[col]): return None
    px   = float(ttf_row[col])
    prev = float(prev_row[col]) if col in prev_row.index and not pd.isna(prev_row[col]) else px
    chg  = px - prev
    pct  = chg / prev * 100 if prev else 0
    sign = '+' if chg >= 0 else ''
    lbl  = delivery_label(rdate, m)
    return f'  M{m} ({lbl:<7}): \u20ac{px:>6.2f}/MWh   {sign}{chg:.2f}  ({sign}{pct:.1f}%)'

# ── Seasonal averages ─────────────────────────────────────────────────────────
# For each of M1-M12, compute which calendar month it delivers in
summer_tenors, winter_tenors = [], []
q4_tenors, q1_tenors = [], []
for m in range(1, 25):
    col = f'M{m}'
    if col not in ttf_row.index or pd.isna(ttf_row[col]): continue
    dm = (rdate.month + m - 1) % 12 + 1
    yr = rdate.year + (rdate.month + m - 1) // 12
    if dm in [4,5,6,7,8,9]:  summer_tenors.append(col)
    if dm in [10,11,12,1,2,3]: winter_tenors.append(col)
    if dm in [10,11,12] and yr == rdate.year + (1 if rdate.month > 9 else 0): q4_tenors.append(col)
    if dm in [1,2,3]    and yr == rdate.year + 1: q1_tenors.append(col)

def avg_tenors(cols):
    vals = [float(ttf_row[c]) for c in cols if c in ttf_row.index and not pd.isna(ttf_row[c])]
    return np.mean(vals) if vals else np.nan

summer_avg = avg_tenors(summer_tenors[:6])
winter_avg = avg_tenors(winter_tenors[:6])
ws_spread  = winter_avg - summer_avg
q4_avg     = avg_tenors(q4_tenors[:3])
q1_avg     = avg_tenors(q1_tenors[:3])

m1_price = float(ttf_row['M1']) if 'M1' in ttf_row.index else np.nan
m12_price = float(ttf_row['M12']) if 'M12' in ttf_row.index else np.nan
m1m12_spread = m1_price - m12_price if not (pd.isna(m1_price) or pd.isna(m12_price)) else np.nan

# ── Storage metrics ───────────────────────────────────────────────────────────
stor_row = storage.loc[storage.index[storage.index <= rdate][-1]] \
           if (storage.index <= rdate).any() else storage.iloc[-1]
fill_pct   = float(stor_row.get('full', np.nan))
cap_twh    = float(stor_row.get('workingGasVolume', np.nan))
cur_twh    = float(stor_row.get('gasInStorage', np.nan))

# WoW fill change
week_ago_idx = storage.index[storage.index <= (rdate - timedelta(days=7))]
fill_wow = fill_pct - float(storage.loc[week_ago_idx[-1], 'full']) if len(week_ago_idx) else np.nan

# Required injection pace to 90% by Nov 1
nov1 = pd.Timestamp(rdate.year if rdate.month < 11 else rdate.year + 1, 11, 1)
days_to_nov1 = (nov1 - rdate).days
target_twh   = cap_twh * 0.90 if not pd.isna(cap_twh) else np.nan
needed_gwh   = (target_twh - cur_twh) * 1000 if not pd.isna(target_twh) else np.nan
req_pace     = needed_gwh / days_to_nov1 if days_to_nov1 > 0 and not pd.isna(needed_gwh) else np.nan

# 21d realized vol
vol_21d = np.nan
if 'M1' in ttf.columns:
    m1_s  = ttf['M1'].loc[ttf.index <= rdate].dropna()
    if len(m1_s) > 22:
        lr    = np.log(m1_s / m1_s.shift(1)).dropna()
        vol_21d = float(lr.rolling(21).std().iloc[-1]) * np.sqrt(252) * 100

# LNG fill
lng_fill = np.nan
if not lng.empty and (lng.index <= rdate).any():
    lng_row  = lng.loc[lng.index[lng.index <= rdate][-1]]
    lng_fill = float(lng_row.get('full', np.nan))

# ── Q4/Q1 label helpers ───────────────────────────────────────────────────────
q4_yr = rdate.year if rdate.month <= 9 else rdate.year + 1
q1_yr = q4_yr + 1

# ── Print dashboard ───────────────────────────────────────────────────────────
W = 52
print('=' * W)
print(f'  TTF DAILY MARKET REPORT \u2014 {rdate.strftime("%d %b %Y")}'.center(W))
print('=' * W)
for m in range(1, 4):
    line = price_line(m)
    if line: print(line)
print()
if not pd.isna(q4_avg):     print(f'  Q4\'{str(q4_yr)[-2:]} avg       : \u20ac{q4_avg:.2f}/MWh')
if not pd.isna(q1_avg):     print(f'  Q1\'{str(q1_yr)[-2:]} avg       : \u20ac{q1_avg:.2f}/MWh')
if not pd.isna(winter_avg): print(f'  Winter avg        : \u20ac{winter_avg:.2f}/MWh')
if not pd.isna(summer_avg): print(f'  Summer avg        : \u20ac{summer_avg:.2f}/MWh')
if not pd.isna(ws_spread):
    shape = 'contango' if ws_spread < 0 else 'backwardation'
    print(f'  W\u2013S spread       : {ws_spread:+.2f} \u20ac/MWh  ({shape})')
if not pd.isna(m1m12_spread):
    shape2 = 'backwardation' if m1m12_spread > 0 else 'contango'
    print(f'  M1\u2013M12 spread    : {m1m12_spread:+.2f} \u20ac/MWh  ({shape2})')
print()
if not pd.isna(fill_pct):   print(f'  EU Storage        : {fill_pct:.1f}%  ({fill_wow:+.1f}pp wow)')
if not pd.isna(req_pace):   print(f'  Required pace     : {req_pace:,.0f} GWh/day to reach 90% by Nov 1')
print(f'  Days to Nov 1     : {days_to_nov1}')
if not pd.isna(lng_fill):   print(f'  LNG Fill          : {lng_fill:.1f}%')
if not pd.isna(vol_21d):    print(f'  21d Realized Vol  : {vol_21d:.1f}% annualised')
print('=' * W)

## 2 · Forward Curve Chart

In [ ]:
def curve_row(ref_date):
    """Return (actual_date, row) for the closest trading day <= ref_date."""
    idx = ttf.index[ttf.index <= ref_date]
    if not len(idx): return None, None
    dt  = idx[-1]
    return dt, ttf.loc[dt]

overlays = [
    ('Today',   rdate,                          '#6366f1', 2.5),
    ('1m ago',  rdate - timedelta(days=30),     '#f97316', 1.5),
    ('3m ago',  rdate - timedelta(days=91),     '#3b82f6', 1.5),
    ('1yr ago', rdate - timedelta(days=365),    '#94a3b8', 1.2),
]

fig = go.Figure()
for label, ref, color, width in overlays:
    dt, row = curve_row(ref)
    if row is None: continue
    xs, ys = [], []
    for m in range(1, 13):
        col = f'M{m}'
        if col not in row.index or pd.isna(row[col]): continue
        dm = (dt.month + m - 1) % 12 + 1
        yr = dt.year + (dt.month + m - 1) // 12
        xs.append(f"{calendar.month_abbr[dm]}'{str(yr)[-2:]}")
        ys.append(float(row[col]))
    if xs:
        dash = 'solid' if label == 'Today' else 'dot'
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode='lines+markers' if label == 'Today' else 'lines',
            name=f'{label} ({dt.strftime("%d %b %Y")})',
            line=dict(color=color, width=width, dash=dash),
            marker=dict(size=5) if label == 'Today' else dict(size=0),
        ))

fig.update_layout(
    template='plotly_white', height=400,
    title=f'TTF Forward Curve M1\u2013M12 ({rdate.date()})',
    yaxis_title='\u20ac/MWh', xaxis_title='Delivery Month',
    xaxis=dict(tickangle=30, tickfont=dict(size=9)),
    legend=dict(orientation='h', y=-0.22),
)
fig.show()
FIG_CURVE = fig

## 3 · Storage Fill + W–S Spread

In [ ]:
lookback_start = rdate - timedelta(days=365)
stor12 = storage.loc[(storage.index >= lookback_start) & (storage.index <= rdate)]

# ── Injection scenarios to Nov 1 ──────────────────────────────────────────────
scen_rates_gwh = {}
if not pd.isna(cur_twh) and not pd.isna(cap_twh) and days_to_nov1 > 0:
    cur_gwh    = cur_twh * 1000
    cap_gwh    = cap_twh * 1000
    # P10/P50/P90 from last 5 years of injection-season daily rates
    inj_hist   = storage['injection'].dropna()
    inj_season = inj_hist[inj_hist.index.month.isin([4,5,6,7,8,9,10])]
    p10 = float(inj_season.quantile(0.10)) if len(inj_season) else 0
    p50 = float(inj_season.quantile(0.50)) if len(inj_season) else 0
    p90 = float(inj_season.quantile(0.90)) if len(inj_season) else 0
    scen_rates_gwh = {'Low (P10)': p10, 'Mid (P50)': p50, 'High (P90)': p90}

# ── W-S spread time-series ────────────────────────────────────────────────────
ttf12 = ttf.loc[(ttf.index >= lookback_start) & (ttf.index <= rdate)]
ws_ts = pd.Series(dtype=float, name='ws_spread')
for dt, row in ttf12.iterrows():
    s_cols = [f'M{m}' for m in range(1,13) if (dt.month+m-1)%12+1 in [4,5,6,7,8,9]
              and f'M{m}' in row.index and not pd.isna(row[f'M{m}'])]
    w_cols = [f'M{m}' for m in range(1,13) if (dt.month+m-1)%12+1 in [10,11,12,1,2,3]
              and f'M{m}' in row.index and not pd.isna(row[f'M{m}'])]
    if s_cols and w_cols:
        sv = float(np.mean([row[c] for c in s_cols[:6]]))
        wv = float(np.mean([row[c] for c in w_cols[:6]]))
        ws_ts.loc[dt] = wv - sv

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[
        'EU Storage Fill Rate (%) \u2014 last 12 months',
        'Winter\u2013Summer Spread (\u20ac/MWh)  |  +ve = Winter premium',
    ],
    vertical_spacing=0.10, row_heights=[0.55, 0.45],
)

# Fill rate
if 'full' in stor12.columns:
    fig.add_trace(go.Scatter(
        x=stor12.index, y=stor12['full'], name='EU Fill %',
        fill='tozeroy', line=dict(color='#22c55e', width=1.8),
        fillcolor='rgba(34,197,94,0.12)',
    ), row=1, col=1)
    fig.add_hline(y=90, line_dash='dot', line_color='purple',
                  annotation_text='90% target', row=1, col=1)

# Injection scenario lines
scen_colors = {'Low (P10)': '#ef4444', 'Mid (P50)': '#3b82f6', 'High (P90)': '#22c55e'}
if scen_rates_gwh and not pd.isna(cur_twh) and not pd.isna(cap_twh):
    dates_proj = [rdate + timedelta(days=d) for d in range(days_to_nov1 + 1)]
    for name, rate_gwh in scen_rates_gwh.items():
        proj_fill = []
        gwh = cur_twh * 1000
        cap_gwh2 = cap_twh * 1000
        for _ in dates_proj:
            proj_fill.append(min(gwh / cap_gwh2 * 100, 100))
            gwh = min(gwh + rate_gwh, cap_gwh2)
        fig.add_trace(go.Scatter(
            x=dates_proj, y=proj_fill, name=name,
            line=dict(color=scen_colors[name], width=1.5, dash='dash'),
        ), row=1, col=1)

# W-S spread
if not ws_ts.empty:
    pos = ws_ts.clip(lower=0); neg = ws_ts.clip(upper=0)
    fig.add_trace(go.Scatter(
        x=ws_ts.index, y=pos, fill='tozeroy', name='Winter premium',
        line=dict(color='#f97316', width=0.5), fillcolor='rgba(249,115,22,0.25)',
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=ws_ts.index, y=neg, fill='tozeroy', name='Summer premium',
        line=dict(color='#3b82f6', width=0.5), fillcolor='rgba(59,130,246,0.25)',
    ), row=2, col=1)
    fig.add_hline(y=0,  line_color='black', line_width=0.8, row=2, col=1)
    fig.add_hline(y=5,  line_dash='dot', line_color='green',
                  annotation_text='\u20ac5 breakeven', row=2, col=1)

fig.update_yaxes(title_text='Fill %',  row=1, col=1)
fig.update_yaxes(title_text='\u20ac/MWh', row=2, col=1)
fig.update_layout(
    template='plotly_white', height=500,
    title='EU Storage Fill & Winter\u2013Summer Spread',
    legend=dict(orientation='h', y=-0.14),
)
fig.show()
FIG_STORAGE = fig

## 4 · Volatility Snapshot

In [ ]:
if 'M1' not in ttf.columns:
    print('\u26a0\ufe0f  M1 column not in TTF data')
    FIG_VOL = go.Figure()
else:
    m1_full = ttf['M1'].dropna()
    lr      = np.log(m1_full / m1_full.shift(1)).dropna()
    vol21   = lr.rolling(21).std()  * np.sqrt(252) * 100
    vol63   = lr.rolling(63).std()  * np.sqrt(252) * 100

    # Last 12 months
    v21_12  = vol21.loc[(vol21.index >= lookback_start) & (vol21.index <= rdate)]
    v63_12  = vol63.loc[(vol63.index >= lookback_start) & (vol63.index <= rdate)]

    cur_21  = float(vol21.loc[vol21.index <= rdate].iloc[-1])
    cur_63  = float(vol63.loc[vol63.index <= rdate].iloc[-1])
    avg_21  = float(vol21.mean())
    med_21  = float(vol21.median())

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=v21_12.index, y=v21_12, name='21d vol',
        fill='tozeroy', line=dict(color='#f97316', width=1.8),
        fillcolor='rgba(249,115,22,0.10)',
    ))
    fig.add_trace(go.Scatter(
        x=v63_12.index, y=v63_12, name='63d vol',
        line=dict(color='#6366f1', width=1.5, dash='dot'),
    ))
    fig.add_hline(y=cur_21, line_color='#f97316', line_dash='dash', line_width=0.8,
                  annotation_text=f'Current 21d: {cur_21:.1f}%',
                  annotation_position='top right')
    fig.add_hline(y=avg_21, line_color='#94a3b8', line_dash='dot', line_width=0.8,
                  annotation_text=f'1yr avg: {avg_21:.1f}%',
                  annotation_position='bottom right')
    fig.add_hline(y=med_21, line_color='#64748b', line_dash='longdash', line_width=0.8,
                  annotation_text=f'Hist. median: {med_21:.1f}%',
                  annotation_position='bottom left')

    fig.update_layout(
        template='plotly_white', height=350,
        title=f'TTF M1 Realised Volatility (annualised %) \u2014 last 12 months',
        yaxis_title='Vol %',
        legend=dict(orientation='h', y=-0.18),
    )
    fig.show()
    FIG_VOL = fig

    # Percentile rank
    pct_rank = float(pd.Series(vol21.dropna()).rank(pct=True).iloc[-1] * 100)
    print(f'21d vol: {cur_21:.1f}%  (p{pct_rank:.0f} vs full history)')
    print(f'63d vol: {cur_63:.1f}%')
    print(f'1yr avg 21d vol: {avg_21:.1f}%  |  historical median: {med_21:.1f}%')

## 5 · Key Levels Table

In [ ]:
def _ttf_val(m_col, ref_date):
    idx = ttf.index[ttf.index <= ref_date]
    if not len(idx) or m_col not in ttf.columns: return np.nan
    v = ttf.loc[idx[-1], m_col]
    return float(v) if not pd.isna(v) else np.nan

def _stor_val(col, ref_date):
    idx = storage.index[storage.index <= ref_date]
    if not len(idx) or col not in storage.columns: return np.nan
    v = storage.loc[idx[-1], col]
    return float(v) if not pd.isna(v) else np.nan

def _vol21(ref_date):
    if 'M1' not in ttf.columns: return np.nan
    s = ttf['M1'].loc[ttf.index <= ref_date].dropna()
    if len(s) < 22: return np.nan
    lr = np.log(s / s.shift(1)).dropna()
    v  = lr.rolling(21).std().iloc[-1]
    return float(v) * np.sqrt(252) * 100 if not pd.isna(v) else np.nan

def _ws(ref_date):
    idx = ttf.index[ttf.index <= ref_date]
    if not len(idx): return np.nan
    dt  = idx[-1]; row = ttf.loc[dt]
    s_v = [float(row[f'M{m}']) for m in range(1,13)
           if (dt.month+m-1)%12+1 in [4,5,6,7,8,9]
           and f'M{m}' in row.index and not pd.isna(row[f'M{m}'])]
    w_v = [float(row[f'M{m}']) for m in range(1,13)
           if (dt.month+m-1)%12+1 in [10,11,12,1,2,3]
           and f'M{m}' in row.index and not pd.isna(row[f'M{m}'])]
    if s_v and w_v: return np.mean(w_v[:6]) - np.mean(s_v[:6])
    return np.nan

def _m1m12(ref_date):
    return _ttf_val('M1', ref_date) - _ttf_val('M12', ref_date)

refs = {
    'Current':  rdate,
    '1W ago':   rdate - timedelta(days=7),
    '1M ago':   rdate - timedelta(days=30),
    '1Y ago':   rdate - timedelta(days=365),
}

def _fmt(v, fmt=':.2f', suffix='', na='—'):
    return f'{v{fmt}}{suffix}' if not pd.isna(v) else na

rows = [
    ('M1 price (\u20ac/MWh)',   [_ttf_val('M1', d)            for d in refs.values()]),
    ('M12 price (\u20ac/MWh)',  [_ttf_val('M12', d)           for d in refs.values()]),
    ('M1\u2013M12 spread',      [_m1m12(d)                    for d in refs.values()]),
    ('W\u2013S spread',         [_ws(d)                       for d in refs.values()]),
    ('EU fill %',               [_stor_val('full', d)         for d in refs.values()]),
    ('21d Realized Vol',        [_vol21(d)                    for d in refs.values()]),
]

col_w   = [22] + [10] * len(refs)
sep     = '+' + '+'.join('-'*w for w in col_w) + '+'
hdr     = '|' + '|'.join(f'{h:^{w}}' for h, w in zip(['Metric'] + list(refs.keys()), col_w)) + '|'
print(sep); print(hdr); print(sep)
for label, vals in rows:
    cells = [f'{label:<{col_w[0]}}'] + [
        f'{v:>{col_w[i+1]}.2f}' if not pd.isna(v) else f'{"\u2014":>{col_w[i+1]}}'
        for i, v in enumerate(vals)
    ]
    print('|' + '|'.join(cells) + '|')
print(sep)

## 6 · PDF Export

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.colors import HexColor, white
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak,
    Table, TableStyle, HRFlowable, Image,
)
from reportlab.lib.enums import TA_CENTER, TA_RIGHT
import tempfile

# ── Palette (matches notebook 07 style) ──────────────────────────────────────
NAVY     = HexColor('#0f172a'); BLUE    = HexColor('#2E75B6')
BLUE_LT  = HexColor('#dbeafe'); BLUE_ROW= HexColor('#EBF3FB')
GREY_BDR = HexColor('#e2e8f0'); SLATE   = HexColor('#64748b')
CHARCOAL = HexColor('#1e293b'); GREEN   = HexColor('#166534')
RED_D    = HexColor('#991b1b')

W_PT, H_PT = letter
MARGIN      = 0.75 * inch
CW          = W_PT - 2 * MARGIN

ss = getSampleStyleSheet()
def _sty(name, **kw): return ParagraphStyle(name, parent=ss['Normal'], **kw)
S = {
    'h1':   _sty('h1',  fontName='Helvetica-Bold', fontSize=14, textColor=BLUE,
                        spaceBefore=10, spaceAfter=4, leading=18),
    'h2':   _sty('h2',  fontName='Helvetica-Bold', fontSize=11, textColor=CHARCOAL,
                        spaceBefore=8,  spaceAfter=3, leading=14),
    'body': _sty('body',fontName='Helvetica',      fontSize=9,  textColor=CHARCOAL,
                        spaceBefore=2,  spaceAfter=3, leading=12),
    'mono': _sty('mono',fontName='Courier',        fontSize=8.5,textColor=CHARCOAL,
                        spaceBefore=1,  spaceAfter=1, leading=11),
    'cell': _sty('cell',fontName='Helvetica',      fontSize=8.5,textColor=CHARCOAL, leading=11),
    'hdr':  _sty('hdr', fontName='Helvetica-Bold', fontSize=8.5,
                        textColor=HexColor('#1e3a5f'), leading=11),
    'ct':   _sty('ct',  fontName='Helvetica-Bold', fontSize=26, textColor=white,
                        alignment=TA_CENTER, leading=32),
    'cs':   _sty('cs',  fontName='Helvetica',      fontSize=11, textColor=HexColor('#94a3b8'),
                        alignment=TA_CENTER, leading=14),
    'cm':   _sty('cm',  fontName='Helvetica',      fontSize=9,  textColor=HexColor('#64748b'),
                        alignment=TA_CENTER, leading=12),
}

def sp(n=8):  return Spacer(1, n)
def hr():     return HRFlowable(width='100%', thickness=0.5, color=GREY_BDR,
                                 spaceAfter=5, spaceBefore=5)
def h1(t):    return [Paragraph(t, S['h1']),
                      HRFlowable(width='100%', thickness=1.5, color=BLUE,
                                 spaceAfter=6, spaceBefore=2)]
def p(t):     return Paragraph(t, S['body'])
def mono(t):  return Paragraph(t.replace(' ',' ').replace('\n','<br/>'), S['mono'])

def tbl(headers, rows_data, widths):
    data = [[Paragraph(h, S['hdr']) for h in headers]]
    for ri, row_d in enumerate(rows_data):
        data.append([Paragraph(str(c), S['cell']) for c in row_d])
    t = Table(data, colWidths=widths)
    cmds = [
        ('BACKGROUND', (0,0), (-1,0), BLUE_LT),
        ('GRID',       (0,0), (-1,-1), 0.4, GREY_BDR),
        ('TOPPADDING', (0,0), (-1,-1), 4),
        ('BOTTOMPADDING', (0,0), (-1,-1), 4),
        ('LEFTPADDING',   (0,0), (-1,-1), 6),
        ('RIGHTPADDING',  (0,0), (-1,-1), 6),
        ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ]
    for ri in range(1, len(rows_data)+1):
        if ri % 2 == 0: cmds.append(('BACKGROUND', (0,ri), (-1,ri), BLUE_ROW))
    t.setStyle(TableStyle(cmds))
    return [t, sp(8)]

FIGS_PDF = {}
def save_fig_pdf(fig, name, w=680, h=360):
    tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    fig.write_image(tmp.name, width=w, height=h, scale=2)
    FIGS_PDF[name] = tmp.name

def img(name, width=CW, height=3.0*inch):
    return [Image(FIGS_PDF[name], width=width, height=height), sp(6)]

def on_page(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(NAVY)
    canvas.rect(MARGIN, H_PT - 0.50*inch, CW, 0.28*inch, fill=1, stroke=0)
    canvas.setFillColor(white); canvas.setFont('Helvetica-Bold', 7.5)
    canvas.drawString(MARGIN+6, H_PT - 0.37*inch, 'TTF Daily Market Report')
    canvas.setFont('Helvetica', 7.5)
    canvas.drawRightString(W_PT - MARGIN - 4, H_PT - 0.37*inch,
                           rdate.strftime('%d %B %Y'))
    canvas.setStrokeColor(GREY_BDR); canvas.setLineWidth(0.4)
    canvas.line(MARGIN, 0.50*inch, W_PT - MARGIN, 0.50*inch)
    canvas.setFillColor(SLATE); canvas.setFont('Helvetica', 7.5)
    canvas.drawRightString(W_PT - MARGIN, 0.35*inch, f'Page {doc.page}')
    canvas.restoreState()

def on_first(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(NAVY)
    canvas.rect(0, H_PT - 2.4*inch, W_PT, 2.4*inch, fill=1, stroke=0)
    canvas.setFillColor(BLUE)
    canvas.rect(0, H_PT - 2.4*inch, W_PT, 0.04*inch, fill=1, stroke=0)
    canvas.restoreState()

# ── Save chart PNGs ───────────────────────────────────────────────────────────
save_fig_pdf(FIG_CURVE,   'curve',   h=320)
save_fig_pdf(FIG_STORAGE, 'storage', h=400)
save_fig_pdf(FIG_VOL,     'vol',     h=280)
print(f'\u2705 {len(FIGS_PDF)} chart PNGs rendered')

# ── Build PDF ─────────────────────────────────────────────────────────────────
Path('data/processed').mkdir(parents=True, exist_ok=True)
PDF_PATH = Path(f'data/processed/ttf_daily_report_{rdate.strftime("%Y%m%d")}.pdf')

doc = SimpleDocTemplate(
    str(PDF_PATH),
    pagesize=letter,
    leftMargin=MARGIN, rightMargin=MARGIN,
    topMargin=0.70*inch, bottomMargin=0.65*inch,
)

story = []

# ── Cover ─────────────────────────────────────────────────────────────────────
m1_label = delivery_label(rdate, 1) if 'M1' in ttf_row.index else ''
m1_str   = f'M1 ({m1_label}) \u20ac{m1_price:.2f}/MWh' if not pd.isna(m1_price) else ''
fill_str = f'EU Fill {fill_pct:.1f}%' if not pd.isna(fill_pct) else ''

story += [
    sp(2.5*inch),
    Paragraph('TTF Daily Market Report', S['ct']),
    sp(10),
    Paragraph(rdate.strftime('%d %B %Y'), S['cs']),
    sp(6),
    Paragraph(f'{m1_str}  ·  {fill_str}', S['cm']),
    sp(18),
    HRFlowable(width='25%', thickness=1, color=BLUE, spaceAfter=10, hAlign='CENTER'),
    Paragraph('Forward Curve · Storage · Volatility · Key Levels', S['cm']),
    PageBreak(),
]

# ── Snapshot table ────────────────────────────────────────────────────────────
story += h1('Market Snapshot')

def _fv(v, fmt='.2f', na='—'): return f'{v:{fmt}}' if not pd.isna(v) else na
def _chg(cur, prev):
    if pd.isna(cur) or pd.isna(prev): return '—'
    d = cur - prev; pct = d/prev*100 if prev else 0
    s = '+' if d >= 0 else ''
    return f'{s}{d:.2f} ({s}{pct:.1f}%)'

snap_rows = []
for m in [1, 2, 3]:
    col = f'M{m}'
    if col not in ttf_row.index: continue
    cx = float(ttf_row[col]) if not pd.isna(ttf_row[col]) else np.nan
    px = float(prev_row[col]) if col in prev_row.index and not pd.isna(prev_row[col]) else np.nan
    snap_rows.append([f'M{m} ({delivery_label(rdate, m)})',
                      f'\u20ac{_fv(cx)}/MWh', _chg(cx, px)])

snap_rows += [
    [f'Q4\'{str(q4_yr)[-2:]} avg',  f'\u20ac{_fv(q4_avg)}/MWh', ''],
    [f'Q1\'{str(q1_yr)[-2:]} avg',  f'\u20ac{_fv(q1_avg)}/MWh', ''],
    ['Winter avg (6m)',             f'\u20ac{_fv(winter_avg)}/MWh', ''],
    ['Summer avg (6m)',             f'\u20ac{_fv(summer_avg)}/MWh', ''],
    ['W\u2013S spread',             f'{_fv(ws_spread):>+7}/MWh',
     'contango' if not pd.isna(ws_spread) and ws_spread < 0 else 'winter premium'],
    ['M1\u2013M12 spread',         f'{_fv(m1m12_spread):>+7}/MWh',
     'backwardation' if not pd.isna(m1m12_spread) and m1m12_spread > 0 else 'contango'],
]
story += tbl(['Metric', 'Price', 'Change / Shape'],
             snap_rows,
             [2.6*inch, 1.8*inch, 2.4*inch])

stor_rows = [
    ['EU fill rate', f'{_fv(fill_pct, ".1f")}%', f'{fill_wow:+.1f}pp wow' if not pd.isna(fill_wow) else '—'],
    ['Required injection pace',
     f'{req_pace:,.0f} GWh/day' if not pd.isna(req_pace) else '—',
     f'to reach 90% by Nov 1 ({days_to_nov1}d)'],
    ['LNG fill rate',
     f'{_fv(lng_fill, ".1f")}%' if not pd.isna(lng_fill) else 'N/A', ''],
    ['21d Realized Vol',
     f'{_fv(vol_21d, ".1f")}% ann.' if not pd.isna(vol_21d) else '—', ''],
]
story += h1('Fundamentals')
story += tbl(['Metric', 'Value', 'Note'],
             stor_rows,
             [2.6*inch, 1.8*inch, 2.4*inch])

story.append(PageBreak())

# ── Forward curve ─────────────────────────────────────────────────────────────
story += h1('Forward Curve M1\u2013M12')
story += img('curve', height=3.0*inch)
story.append(p(
    f'Tenor M1 = {delivery_label(rdate, 1)} delivery at \u20ac{_fv(m1_price)}/MWh. '
    f'M1\u2013M12 spread: {_fv(m1m12_spread, "+.2f")}/MWh. '
    f'Dashed lines show the curve 1 month, 3 months, and 1 year ago.'
))
story.append(PageBreak())

# ── Storage + spreads ─────────────────────────────────────────────────────────
story += h1('EU Storage Fill & W\u2013S Spread')
story += img('storage', height=3.8*inch)
story.append(p(
    f'EU fill rate: {_fv(fill_pct, ".1f")}% as of {rdate.date()}. '
    f'Required injection pace: {req_pace:,.0f} GWh/day to reach 90%% by Nov 1 ' if not pd.isna(req_pace) else ''
    f'({days_to_nov1} days remaining). '
    f'W\u2013S spread: {_fv(ws_spread, "+.2f")}/MWh '
    f'({"above" if not pd.isna(ws_spread) and ws_spread >= 5 else "below"} \u20ac5 injection breakeven).'
))
story.append(PageBreak())

# ── Volatility ────────────────────────────────────────────────────────────────
story += h1('Realised Volatility')
story += img('vol', height=2.6*inch)
if not pd.isna(vol_21d):
    story.append(p(
        f'21-day realised vol: {vol_21d:.1f}% annualised. '
        f'63-day: {_fv(cur_63 if "cur_63" in dir() else np.nan, ".1f")}%. '
        f'Dashed lines show current level, 1-year average, and historical median.'
    ))
story.append(PageBreak())

# ── Key levels ────────────────────────────────────────────────────────────────
story += h1('Key Levels')
kl_headers = ['Metric', 'Current', '1W ago', '1M ago', '1Y ago']
kl_rows = []
for label, vals in rows:   # 'rows' from cell 5
    kl_rows.append([label] + [_fv(v) for v in vals])
story += tbl(kl_headers, kl_rows, [2.1*inch, 1.1*inch, 1.1*inch, 1.1*inch, 1.1*inch])

# ── Build ─────────────────────────────────────────────────────────────────────
doc.build(story, onFirstPage=on_first, onLaterPages=on_page)
print(f'\n\u2705 PDF saved \u2192 {PDF_PATH}')
print(f'   Size: {PDF_PATH.stat().st_size / 1024:.1f} KB')

from IPython.display import FileLink, display
display(FileLink(str(PDF_PATH), result_html_prefix='\U0001f4c4 Download: '))